# Modelo de Fuga (Compra de Deuda) — NBC Empresas
**Etapa: modelado, evaluacion y backtest out-of-time**

1. Instalacion e imports · 2. Carga de datos · 3. Variables y target · 4. Validacion + VIF · 5. Optuna + seleccion · 6. Evaluacion / significancia / importancia · 7. ROC, deciles y backtest OOT · 8. WOE-IV (diagnostico) · 9. Escalera de propension · 10. PSI y drift · 11. Guardado de artefactos · 12. Perfiles por grupo de score

### 1. Instalación de dependencias

In [ ]:
%pip install -q awswrangler==3.13.0 jupyterlab_execute_time seaborn xgboost==2.1.3 lightgbm statsmodels scikit-learn scorecardpy boto3 catboost interpret optbinning joblib optuna

### 2. Configuración y librerías

In [ ]:
from dateutil.relativedelta import relativedelta
from time import gmtime, strftime
from datetime import datetime 
import random as rn
import joblib
import time
import json
import sys
import os
import gc

# Nube
import awswrangler as wr
import boto3

# Cálculo
import pandas as pd
import numpy as np
import scipy
from scipy.stats import chi2_contingency

# Gráfico
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set(style="whitegrid")

# ML
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
import scorecardpy as sc

# Configuración
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Seeds para reproducibilidad
SEED = 123
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
rn.seed(SEED)


#from utils_campania import *
from feature_selection_v1 import *

In [ ]:
from dateutil.relativedelta import relativedelta
from time import gmtime, strftime
from datetime import datetime
import random as rn
import joblib
import json
import sys
import os
import gc



#calculo
import pandas as pd
import numpy as np
import scipy

#grafico
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set(style="whitegrid")

#Interacciones con output
import warnings
warnings.filterwarnings("ignore")
# warnings.simplefilter(action='ignore', category=FutureWarning)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

gc.collect()
# MODELS
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report

# LightGBM
from lightgbm import LGBMClassifier
from lightgbm.callback import early_stopping, log_evaluation

# XGBoost
import xgboost as xgb
# Optuna para hyperparameter tuning
try:
    import optuna
    from optuna.samplers import TPESampler
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False
    print("  Optuna no disponible. Instalando...")
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "-q"])
    import optuna
    from optuna.samplers import TPESampler
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
# Sklearn - Preprocessing
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

BASE_DIR = os.path.dirname(os.getcwd())
if BASE_DIR not in sys.path: sys.path.append(BASE_DIR)
    
BASE_DIR = os.path.dirname(os.path.dirname(os.getcwd()))
if BASE_DIR not in sys.path: sys.path.append(BASE_DIR)
print("BASE_DIR::: ", BASE_DIR)
#import scorecardpy as sc



SEED = 29082013
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
rn.seed(SEED)


esquema_vpc = 'disc_comercial'
path_ = 's3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/'

### Imports y utilidades del proyecto

In [ ]:
from dateutil.relativedelta import relativedelta
from time import gmtime, strftime
from datetime import datetime 
import random as rn
import joblib
import time
import json
import sys
import os
import gc

# Nube
import awswrangler as wr
import boto3

# Cálculo
import pandas as pd
import numpy as np
import scipy
from scipy.stats import chi2_contingency

# Gráfico
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set(style="whitegrid")

# ML
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
import scorecardpy as sc

# Configuración
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Seeds para reproducibilidad
SEED = 123
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
rn.seed(SEED)


#from utils_campania import *
from feature_selection_v1 import *

In [ ]:
import pandas as pd
import numpy as np
from utils_modelo import (
    validar_datasets,
    analizar_variables,
    calcular_vif,
    optimizar_modelos,
    evaluar_modelos,
    comparar_modelos,
    obtener_importancia,
    analisis_deciles,
    escalera_propension,
analisis_deciles_por_mes,
    plot_optimization_history,
    plot_roc_curves,
    plot_feature_importance
)


In [ ]:
SEED = 42
N_TRIALS_OPTUNA = 60

### 3. Carga de datos (train / valid / test OOT)

In [ ]:
print("\n" + "="*100)
print(" PASO 0: CARGA DE DATOS")
print("="*100)

# Cargar datos
train_final = pd.read_csv('final_train.csv')
valid_final = pd.read_csv('final_valid.csv')
test_final = pd.read_csv('final_test.csv')

print(f"Train: {train_final.shape}")
print(f"Valid: {valid_final.shape}")
print(f"Test:  {test_final.shape}")

In [ ]:
train_final.info(4)

### 4. Variables del modelo y target

In [ ]:
variables_finales = [
'sow_ibk_promedio_6m',
'meses_con_saldo_colocaciones_6m',
'pct_d_saldo_3m',
'meses_con_saldo_colocaciones_12m',
'deuda_ibk_var_pct_6m',
'sow_ibk_meses_mayor_50_pct',
'deuda_ibk_promedio_12m',
'tea_principal',
'sow_ibk_max_12m',
'deuda_noibk_max_12m',
'avg_saldo_competencia_coloc_directas_general_vig_amt_u6m',
'var_abs_colocaciones_6m',
'avg_nro_impulso_normal_u6m',
'meses_desde_desem_min',
'meses_con_saldo_pasivo_12m',
'pct_avance_cuotas',
'tendencia_saldo_impulso_amt',
'tendencia_saldo_ibk_garantias_amt',
'tendencia_saldo_ibk_coloc_directas_amt_general',
'cuota_total',
'avg_saldo_gar_impulso_amt_u6m',
'var_ibk_ytd_amt',
'deuda_total_var_pct_1m',
'deuda_total_var_pct_3m',
'avg_saldo_ibk_coloc_directas_amt_general_u12m',
'avg_saldo_ibk_garantias_amt_u12m'



 ]

target = 'target'

print(f"\n Variables en el modelo: {len(variables_finales)}")

### 5. Validación de nulos

In [ ]:
validar_datasets(train_final, valid_final, test_final, variables_finales)

### 6. Preparación de X e y

In [ ]:
print("\n" + "="*100)
print(" PASO 2: PREPARACIÓN DE X e y (DATOS CRUDOS - SIN NORMALIZACIÓN)")
print("="*100)

X_train = train_final[variables_finales].copy()
y_train = train_final[target].copy()

X_valid = valid_final[variables_finales].copy()
y_valid = valid_final[target].copy()

X_test = test_final[variables_finales].copy()
y_test = test_final[target].copy()

print(" Dimensiones:")
print(f"   X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"   X_valid: {X_valid.shape} | y_valid: {y_valid.shape}")
print(f"   X_test:  {X_test.shape} | y_test:  {y_test.shape}")

print(f"\n Distribución del target:")
print(f"   Train: {y_train.value_counts(normalize=True).round(4).to_dict()}")
print(f"   Valid: {y_valid.value_counts(normalize=True).round(4).to_dict()}")
print(f"   Test:  {y_test.value_counts(normalize=True).round(4).to_dict()}")

### 7. Análisis de variables + VIF

In [ ]:
analisis_vars = analizar_variables(X_train)

# ============================================================================
# PASO 4: ANÁLISIS DE MULTICOLINEALIDAD (VIF)
# ============================================================================

vif_data = calcular_vif(X_train, variables_finales)

### 8. Tuning con Optuna

In [ ]:
resultados_optuna = optimizar_modelos(
    X_train, y_train, 
    X_valid, y_valid,
    n_trials=N_TRIALS_OPTUNA,
    SEED=SEED
)

# Visualizar historia de optimización
plot_optimization_history(resultados_optuna['studies'])

### 9. Evaluación de modelos

In [ ]:
resultados, modelos_entrenados = evaluar_modelos(
    resultados_optuna['modelos'],
    X_train, y_train,
    X_valid, y_valid,
    X_test, y_test
)

### 10. Comparación y selección del mejor modelo

In [ ]:
df_comparacion, mejor_modelo_nombre = comparar_modelos(resultados)

mejor_modelo = modelos_entrenados[mejor_modelo_nombre]

In [ ]:
from utils_significancia_3 import analisis_completo_significancia

# Ejecutar análisis completo
resultados_var = analisis_completo_significancia(
    modelo=mejor_modelo,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test
)

# Variables confirmadas por TODOS los métodos
vars_confirmadas = resultados_var['consolidado']['vars_en_todos']
print(f"\n Variables altamente significativas: {vars_confirmadas}")

### 11. Importancia de variables

In [ ]:
print("\n" + "="*100)
print(" PASO 8: IMPORTANCIA DE VARIABLES")
print("="*100)

df_importance = obtener_importancia(mejor_modelo, X_train, mejor_modelo_nombre)

if df_importance is not None:
    print(f"\n Top 15 Variables más importantes ({mejor_modelo_nombre}):\n")
    print(df_importance.head(15).to_string(index=False))
    
    # Visualizar
    plot_feature_importance(df_importance, top_n=20)

In [ ]:
df_importance

### 12. Curvas ROC

In [ ]:
print("\n" + "="*100)
print(" PASO 9: CURVAS ROC")
print("="*100)

plot_roc_curves(
    y_train, y_valid, y_test,
    resultados[mejor_modelo_nombre]['predicciones_train'],
    resultados[mejor_modelo_nombre]['predicciones_valid'],
    resultados[mejor_modelo_nombre]['predicciones_test'],
    mejor_modelo_nombre
)

### 13. Análisis de deciles

In [ ]:
print("\n" + "="*100)
print(" PASO 10: ANÁLISIS DE DECILES")
print("="*100)

df_deciles_test = analisis_deciles(
    y_test, 
    resultados[mejor_modelo_nombre]['predicciones_test'],
    'Test'
)

### Distribución de predicciones (test)

In [ ]:
pd.Series(resultados[mejor_modelo_nombre]['predicciones_test']).hist()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('matplotlib').setLevel(logging.ERROR)
pd.options.mode.chained_assignment = None

# ============================================================================
# COLORES INTERBANK
# ============================================================================
def rgb_to_mpl(rgb_str):
    nums = rgb_str.replace('rgb(', '').replace(')', '').split(',')
    return tuple([int(n)/255 for n in nums])

COLOR_VERDE_IBK = rgb_to_mpl('rgb(0,208,60)')
COLOR_AZUL_IBK = rgb_to_mpl('rgb(31,69,146)')
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Poppins', 'Arial', 'Helvetica']

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================
def format_number(val, is_amount=False):
    if pd.isna(val): return "0"
    if is_amount:
        if abs(val) >= 1_000_000: return f"{val/1_000_000:.1f}M"
        elif abs(val) >= 1_000: return f"{val/1_000:.0f}K"
        else: return f"{val:.0f}"
    else:
        return f"{val:,.0f}" if abs(val) >= 1_000 else f"{val:.0f}"

def is_amount_variable(var_name):
    keywords = ['amt','saldo','deuda','monto','importe','balance','capital','credito','coloc','oferta','sldtot']
    return any(k in var_name.lower() for k in keywords)

def clasificar_iv(iv):
    if iv < 0.02: return "No predictivo"
    elif iv < 0.1: return "Débil"
    elif iv < 0.3: return "Medio"
    elif iv < 0.5: return "Fuerte"
    else: return "Muy fuerte"

def merge_small_bins(woe_df, min_pct=5.0):
    total_registros = woe_df['total_registros'].sum()
    min_registros = total_registros * (min_pct / 100)
    if woe_df['total_registros'].min() >= min_registros: return woe_df
    df_merged = woe_df.copy()
    while True:
        min_idx = df_merged['total_registros'].idxmin()
        if df_merged.loc[min_idx, 'total_registros'] >= min_registros: break
        if len(df_merged) <= 1: break
        if min_idx == 0: merge_with = min_idx + 1
        elif min_idx == len(df_merged) - 1: merge_with = min_idx - 1
        else:
            merge_with = min_idx - 1 if df_merged.loc[min_idx-1,'total_registros'] < df_merged.loc[min_idx+1,'total_registros'] else min_idx+1
        idx1, idx2 = sorted([min_idx, merge_with])
        new_row = {
            'bin': f"{df_merged.loc[idx1,'bin']}-{df_merged.loc[idx2,'bin']}",
            'total_registros': df_merged.loc[idx1,'total_registros'] + df_merged.loc[idx2,'total_registros'],
            'eventos': df_merged.loc[idx1,'eventos'] + df_merged.loc[idx2,'eventos'],
            'valor_min': min(df_merged.loc[idx1,'valor_min'], df_merged.loc[idx2,'valor_min']),
            'valor_max': max(df_merged.loc[idx1,'valor_max'], df_merged.loc[idx2,'valor_max'])
        }
        df_merged = df_merged.drop([idx1, idx2])
        df_merged = pd.concat([df_merged, pd.DataFrame([new_row])], ignore_index=True)
        df_merged = df_merged.sort_values('valor_min').reset_index(drop=True)
    return df_merged

def calculate_woe_iv_auto(df, feature_col, target_col, n_bins=5, min_bin_pct=10.0, custom_bins=None):
    try:
        temp_df = df[[feature_col, target_col]].copy().dropna()
        if len(temp_df) == 0: return None, 0, None
        n_unique = temp_df[feature_col].nunique()
        if n_unique < 2: return None, 0, None
        
        is_amt = is_amount_variable(feature_col)
        
        # ── Estrategia zero-split para cualquier variable con muchos ceros ──
        pct_zeros = (temp_df[feature_col] == 0).mean()
        if pct_zeros > 0.30 and n_unique > 2:
            # Separar ceros de no-ceros
            df_zero = temp_df[temp_df[feature_col] == 0].copy()
            df_nonzero = temp_df[temp_df[feature_col] != 0].copy()
            n_nz_unique = df_nonzero[feature_col].nunique()
            target_nz_bins = max(1, min(3, n_nz_unique))  # hasta 3 bins no-cero → 4 total
            
            if len(df_nonzero) > 0 and n_nz_unique >= 2:
                try:
                    df_nonzero['bin'], bins_edges = pd.qcut(df_nonzero[feature_col], q=target_nz_bins, duplicates='drop', labels=False, retbins=True)
                except:
                    df_nonzero['bin'], bins_edges = pd.cut(df_nonzero[feature_col], bins=target_nz_bins, duplicates='drop', labels=False, retbins=True)
                # Offset bins +1 para dejar bin 0 = "ceros"
                df_nonzero['bin'] = df_nonzero['bin'] + 1
            else:
                df_nonzero['bin'] = 1
            
            df_zero['bin'] = 0
            temp_df = pd.concat([df_zero, df_nonzero])
            binning_type = 'zero_split'
        elif custom_bins is not None:
            temp_df['bin'] = pd.cut(temp_df[feature_col], bins=custom_bins, labels=False, include_lowest=True, duplicates='drop')
            binning_type = 'custom'
        elif n_unique <= 10:
            temp_df['bin'] = temp_df[feature_col].astype(str)
            binning_type = 'categorical'
        else:
            effective_bins = 4 if is_amt else n_bins
            try:
                temp_df['bin'], _ = pd.qcut(temp_df[feature_col], q=min(effective_bins, n_unique), duplicates='drop', labels=False, retbins=True)
                binning_type = 'quantile'
            except:
                temp_df['bin'], _ = pd.cut(temp_df[feature_col], bins=min(effective_bins, n_unique), duplicates='drop', labels=False, retbins=True)
                binning_type = 'equal_width'
        
        woe_df = temp_df.groupby('bin', observed=False).agg(
            total_registros=('bin','count'), eventos=(target_col,'sum'),
            valor_min=(feature_col,'min'), valor_max=(feature_col,'max')
        ).reset_index()
        woe_df = woe_df[woe_df['total_registros'] > 0].copy()
        if len(woe_df) < 2: return None, 0, None
        
        if binning_type in ['quantile','equal_width'] and custom_bins is None:
            woe_df = merge_small_bins(woe_df, min_pct=min_bin_pct)
        
        woe_df['no_eventos'] = woe_df['total_registros'] - woe_df['eventos']
        woe_df['tasa_pct'] = (woe_df['eventos'] / woe_df['total_registros'] * 100)
        total_ev = woe_df['eventos'].sum()
        total_noev = woe_df['no_eventos'].sum()
        if total_ev == 0 or total_noev == 0: return None, 0, None
        
        woe_df['dist_ev'] = woe_df['eventos'] / total_ev
        woe_df['dist_noev'] = woe_df['no_eventos'] / total_noev
        eps = 0.0001
        woe_df['woe'] = np.log((woe_df['dist_ev']+eps)/(woe_df['dist_noev']+eps))
        woe_df['iv'] = (woe_df['dist_ev'] - woe_df['dist_noev']) * woe_df['woe']
        woe_df['woe'] = woe_df['woe'].replace([np.inf,-np.inf],0).fillna(0)
        woe_df['iv'] = woe_df['iv'].replace([np.inf,-np.inf],0).fillna(0)
        
        if binning_type == 'zero_split':
            labels = []
            for _, row in woe_df.iterrows():
                if row['valor_min'] == 0 and row['valor_max'] == 0:
                    labels.append('= 0')
                else:
                    labels.append(f"({format_number(row['valor_min'], True)}, {format_number(row['valor_max'], True)}]")
            woe_df['bin_label'] = labels
        elif binning_type in ['quantile','equal_width','custom']:
            if is_amt:
                woe_df['bin_label'] = woe_df.apply(lambda x: f"[{format_number(x['valor_min'],True)}, {format_number(x['valor_max'],True)}]", axis=1)
            else:
                woe_df['bin_label'] = woe_df.apply(lambda x: f"[{x['valor_min']:.1f}, {x['valor_max']:.1f}]", axis=1)
        else:
            woe_df['bin_label'] = woe_df['bin'].astype(str)
        
        woe_df['pct_registros'] = (woe_df['total_registros']/woe_df['total_registros'].sum()*100).round(1)
        resultado = woe_df[['bin_label','total_registros','pct_registros','eventos','tasa_pct','woe','iv']].copy()
        resultado.columns = ['Rango','Total','% Total','Eventos','Tasa_%','WOE','IV']
        resultado['Tasa_%'] = resultado['Tasa_%'].round(2)
        resultado['WOE'] = resultado['WOE'].round(4)
        resultado['IV'] = resultado['IV'].round(4)
        return resultado, woe_df['iv'].sum(), binning_type
    except:
        return None, 0, None

# ============================================================================
# VISUALIZACIÓN WOE
# ============================================================================
def plot_woe_detail(woe_df, variable_name, num_var, total_vars):
    plt.ioff()
    fig = plt.figure(figsize=(10, 15))
    gs = fig.add_gridspec(2, 1, hspace=0.5)
    fig.patch.set_facecolor('white')
    x_pos = np.arange(len(woe_df))
    
    # Panel 1: Distribución + Tasa
    ax1 = fig.add_subplot(gs[0,0]); ax1.set_facecolor('white')
    bars = ax1.bar(x_pos, woe_df['Total'], width=0.7, color=COLOR_VERDE_IBK, alpha=0.85, edgecolor=COLOR_VERDE_IBK, linewidth=1.5)
    for bar, val, pct in zip(bars, woe_df['Total'], woe_df['% Total']):
        h = bar.get_height()
        if h > 0:
            ax1.text(bar.get_x()+bar.get_width()/2., h*0.5, f'{int(val):,}\n({pct}%)', ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    ax1_tw = ax1.twinx()
    ax1_tw.plot(x_pos, woe_df['Tasa_%'], color=COLOR_AZUL_IBK, marker='o', markersize=12, linewidth=4, zorder=5)
    for x, y in zip(x_pos, woe_df['Tasa_%']):
        ax1_tw.annotate(f'{y:.1f}%', xy=(x,y), xytext=(0,15), textcoords='offset points', ha='center', fontsize=12, fontweight='bold', color=COLOR_AZUL_IBK,
                        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor=COLOR_AZUL_IBK, linewidth=2))
    tasa_prom = woe_df['Eventos'].sum()/woe_df['Total'].sum()*100
    ax1_tw.axhline(y=tasa_prom, color=COLOR_AZUL_IBK, linestyle='--', alpha=0.5, linewidth=2.5)
    ax1_tw.text(len(woe_df)-0.5, tasa_prom, f'Tasa Promedio: {tasa_prom:.2f}%', fontsize=11, fontweight='bold', color=COLOR_AZUL_IBK, ha='right', va='bottom',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=COLOR_AZUL_IBK, linewidth=1.5, alpha=0.9))
    ax1.set_xlabel('Rango', fontsize=15, fontweight='bold', color=COLOR_AZUL_IBK)
    ax1.set_ylabel('Total Registros', fontsize=15, fontweight='bold', color=COLOR_VERDE_IBK)
    ax1_tw.set_ylabel('Tasa Evento %', fontsize=15, fontweight='bold', color=COLOR_AZUL_IBK)
    ax1.set_xticks(x_pos); ax1.set_xticklabels(woe_df['Rango'], rotation=45, ha='right', fontsize=12)
    ax1.set_title('Distribución y Tasa de Evento (Target)', fontsize=16, fontweight='bold', color=COLOR_AZUL_IBK, pad=20)
    ax1.grid(True, axis='y', alpha=0.3, linestyle='--'); ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False); ax1_tw.spines['top'].set_visible(False)
    
    # Panel 2: WOE
    ax2 = fig.add_subplot(gs[1,0]); ax2.set_facecolor('white')
    colors_woe = [COLOR_VERDE_IBK if w >= 0 else COLOR_AZUL_IBK for w in woe_df['WOE']]
    bars2 = ax2.bar(x_pos, woe_df['WOE'], width=0.7, color=colors_woe, alpha=0.85, edgecolor='black', linewidth=1.5)
    for bar, wv, iv in zip(bars2, woe_df['WOE'], woe_df['IV']):
        h = bar.get_height()
        if abs(h) > 0.01:
            yp = h + (0.12 if h >= 0 else -0.12)
            ax2.text(bar.get_x()+bar.get_width()/2, yp, f'WOE: {wv:.3f}\nIV: {iv:.4f}', ha='center', va='bottom' if h>=0 else 'top',
                     fontsize=11, fontweight='bold', color=COLOR_AZUL_IBK, bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=COLOR_AZUL_IBK, linewidth=1.5, alpha=0.9))
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=2)
    ax2.set_xlabel('Rango', fontsize=15, fontweight='bold', color=COLOR_AZUL_IBK)
    ax2.set_ylabel('WOE', fontsize=15, fontweight='bold', color=COLOR_AZUL_IBK)
    ax2.set_xticks(x_pos); ax2.set_xticklabels(woe_df['Rango'], rotation=45, ha='right', fontsize=12)
    total_iv = woe_df['IV'].sum()
    ax2.set_title(f'Weight of Evidence (WOE) | IV Total: {total_iv:.4f} ({clasificar_iv(total_iv)})', fontsize=16, fontweight='bold', color=COLOR_AZUL_IBK, pad=20)
    ax2.grid(True, axis='y', alpha=0.3, linestyle='--'); ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
    
    plt.suptitle(f'[{num_var}/{total_vars}] Análisis WOE - {variable_name}', fontsize=18, fontweight='bold', color=COLOR_AZUL_IBK, y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.99]); plt.show(); plt.close('all')

def plot_iv_ranking_all(summary_df):
    plt.ioff()
    fig, ax = plt.subplots(figsize=(16, max(12, len(summary_df) * 0.45)))
    fig.patch.set_facecolor('white'); ax.set_facecolor('white')
    plot_data = summary_df.sort_values('iv', ascending=True).copy()
    color_map = {'Muy fuerte': COLOR_VERDE_IBK, 'Fuerte': (0.4,0.7,0.4), 'Medio': (1.0,0.7,0.0), 'Débil': (1.0,0.4,0.4), 'No predictivo': (0.7,0.7,0.7)}
    colors = [color_map.get(p, (0.8,0.8,0.8)) for p in plot_data['poder_predictivo']]
    bars = ax.barh(range(len(plot_data)), plot_data['iv'], color=colors, edgecolor='black', linewidth=1, alpha=0.85)
    for i, (bar, iv_val) in enumerate(zip(bars, plot_data['iv'])):
        if bar.get_width() > 0.005:
            ax.text(bar.get_width()+0.008, bar.get_y()+bar.get_height()/2, f'{iv_val:.4f}', va='center', fontsize=10, fontweight='bold', color=COLOR_AZUL_IBK)
    ax.axvline(x=0.02, color='gray', linestyle='--', alpha=0.4); ax.axvline(x=0.1, color='red', linestyle='--', alpha=0.5); ax.axvline(x=0.3, color='orange', linestyle='--', alpha=0.5)
    ax.set_yticks(range(len(plot_data))); ax.set_yticklabels(plot_data['variable'], fontsize=11)
    ax.set_xlabel('Information Value (IV)', fontsize=15, fontweight='bold', color=COLOR_AZUL_IBK)
    ax.set_title(f'Ranking Variables por IV | Target: {target} | {len(plot_data)} variables', fontsize=16, fontweight='bold', color=COLOR_AZUL_IBK, pad=20)
    ax.grid(axis='x', alpha=0.3, linestyle='--'); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout(); plt.show(); plt.close('all')

# ============================================================================
# EJECUCIÓN: WOE para las 25 variables del modelo
# ============================================================================
print(f"\n{'='*100}")
print(f"ANÁLISIS WOE - {len(variables_finales)} VARIABLES DEL MODELO vs {target.upper()}")
print(f"Saldos/Montos: estrategia zero-split (=0 | bajo | medio | alto) → 4 rangos")
print(f"{'='*100}\n")

resumen_list = []
detalle = {}

for i, var in enumerate(variables_finales, 1):
    print(f"[{i}/{len(variables_finales)}] {var:45s}...", end=' ')
    woe_r, iv_t, bt = calculate_woe_iv_auto(train_final, var, target, n_bins=5, min_bin_pct=10.0)
    if woe_r is not None:
        detalle[var] = woe_r
        resumen_list.append({'variable': var, 'iv': iv_t, 'n_bins': len(woe_r), 'binning_type': bt, 'poder_predictivo': clasificar_iv(iv_t)})
        print(f"✓ IV={iv_t:.4f} ({clasificar_iv(iv_t)}) - {len(woe_r)} bins [{bt}]")
    else:
        print(f"✗ Sin resultado")

summary_df_woe = pd.DataFrame(resumen_list).sort_values('iv', ascending=False).reset_index(drop=True)
summary_df_woe['rank'] = range(1, len(summary_df_woe)+1)

print(f"\n{'='*100}")
print("RANKING INFORMATION VALUE (IV)")
print(f"{'='*100}")
print(summary_df_woe[['rank','variable','iv','poder_predictivo','n_bins','binning_type']].to_string(index=False))

# Ranking gráfico
plot_iv_ranking_all(summary_df_woe)

# Gráficos detallados de cada variable
for i, var in enumerate(summary_df_woe['variable'], 1):
    if var in detalle:
        iv_var = summary_df_woe[summary_df_woe['variable']==var]['iv'].values[0]
        print(f"\n{'─'*100}")
        print(f"[{i}/{len(detalle)}] {var} | IV: {iv_var:.4f} ({clasificar_iv(iv_var)})")
        print(detalle[var].to_string(index=False))
        plot_woe_detail(detalle[var], var, i, len(detalle))

print(f"\n✓ ANÁLISIS WOE COMPLETADO - {len(detalle)}/{len(variables_finales)} variables")

### Diagnóstico: ¿por qué fallan 8 variables?

In [ ]:
vars_fallidas = [v for v in variables_finales if v not in detalle]
print(f"Variables sin WOE ({len(vars_fallidas)}):\n")
for v in vars_fallidas:
    col = train_final[v].dropna()
    n_unique = col.nunique()
    pct_zero = (col == 0).mean() * 100
    pct_na = train_final[v].isna().mean() * 100
    print(f"  {v:40s} | únicos={n_unique:6d} | %ceros={pct_zero:6.2f}% | %NaN={pct_na:5.2f}% | min={col.min():.2f} | max={col.max():.2f}")

### Recargar utils_modelo con el fix del bug de deciles (category → numeric)

In [ ]:
import importlib
import utils_modelo
importlib.reload(utils_modelo)
from utils_modelo import analisis_deciles_por_mes, analisis_deciles, escalera_propension
print("✓ utils_modelo recargado con fix de deciles")

### 14. Análisis de deciles por mes

In [ ]:
print("\n" + "="*100)
print(" PASO 10.5: ANÁLISIS DE DECILES POR MES")
print("="*100)

# Preparar dataframe con periodo, target y score
df_test_con_periodo = test_final[['periodo_campania']].copy()
df_test_con_periodo['target'] = y_test.values
df_test_con_periodo['score'] = resultados[mejor_modelo_nombre]['predicciones_test']

# Ejecutar análisis por periodo
resultados_por_periodo, resumen_periodos = analisis_deciles_por_mes(
    df_con_fecha=df_test_con_periodo,
    col_fecha='periodo_campania',  # Columna con formato YYYYMM
    col_target='target',
    col_score='score',
    dataset_name='Test'
)

### 15. Backtest out-of-time (ejecutivo)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
import numpy as np
plt.ioff()

def rgb_to_mpl(rgb_str):
    nums = rgb_str.replace('rgb(', '').replace(')', '').split(',')
    return tuple([int(n)/255 for n in nums])

COLOR_VERDE = rgb_to_mpl('rgb(0,208,60)')
COLOR_AZUL = rgb_to_mpl('rgb(31,69,146)')
COLORES_PERIODO = [COLOR_VERDE, COLOR_AZUL, (0.9, 0.5, 0.0), (0.6, 0.2, 0.8)]
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Poppins', 'Arial', 'Helvetica']

periodos = sorted(resultados_por_periodo.keys())
label_map = dict(zip(resumen_periodos['periodo'].astype(str), resumen_periodos['anio_mes']))
kpi_map = {str(row['periodo']): row for _, row in resumen_periodos.iterrows()}

# Helper: renumerar deciles para que D10=mejor score
def renumerar_deciles(df_per):
    df = df_per.copy()
    df['decil_display'] = 11 - df['decil']  # D1(mejor) → D10, D10(peor) → D1
    return df.sort_values('decil_display')

# ============================================================================
# GRÁFICO 1: RESUMEN EJECUTIVO (4 KPIs por periodo)
# ============================================================================
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.patch.set_facecolor('white')
fig.suptitle('BACKTEST DE DECILES - RESUMEN EJECUTIVO', fontsize=20, fontweight='bold', color=COLOR_AZUL, y=1.02)

# Nota: en resumen_periodos, "decil_1" = mejor score → lo mostramos como "Decil 10"
kpis = [
    ('KS Máximo', 'ks_max', '{:.1f}'),
    ('Lift Decil 10', 'lift_decil_1', '{:.2f}x'),
    ('Tasa Decil 10', 'efectividad_decil_1', '{:.1f}%'),
    ('Tasa Global', 'efectividad_global', '{:.1f}%')
]

for idx, (titulo, col, fmt) in enumerate(kpis):
    ax = axes[idx]; ax.set_facecolor('white')
    vals = [kpi_map[p][col] for p in periodos]
    x = np.arange(len(periodos))
    bars = ax.bar(x, vals, color=[COLORES_PERIODO[i] for i in range(len(periodos))], width=0.5, edgecolor='black', linewidth=1.2, alpha=0.9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.03, fmt.format(v),
                ha='center', fontsize=16, fontweight='bold', color=COLOR_AZUL)
    ax.set_xticks(x); ax.set_xticklabels([label_map[p] for p in periodos], fontsize=13, fontweight='bold')
    ax.set_title(titulo, fontsize=14, fontweight='bold', color=COLOR_AZUL, pad=10)
    ax.set_ylim(0, max(vals)*1.25)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout(); plt.show(); plt.close()

# ============================================================================
# GRÁFICO 2: TASA DE EVENTO POR DECIL (D1=peor → D10=mejor)
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 8))
fig.patch.set_facecolor('white'); ax.set_facecolor('white')
bar_width = 0.8 / len(periodos)

for i, per in enumerate(periodos):
    df_r = renumerar_deciles(resultados_por_periodo[per])
    tasas = df_r['efectividad_desembolso'].values
    x = np.arange(10) + i * bar_width
    bars = ax.bar(x, tasas, width=bar_width, color=COLORES_PERIODO[i], alpha=0.85, edgecolor='black', linewidth=0.8, label=label_map[per])
    for bar, t in zip(bars, tasas):
        if bar.get_height() > 0:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, f'{t:.1f}%',
                    ha='center', va='bottom', fontsize=9, fontweight='bold', color=COLOR_AZUL)

for i, per in enumerate(periodos):
    tasa_global = kpi_map[per]['efectividad_global']
    ax.axhline(y=tasa_global, color=COLORES_PERIODO[i], linestyle='--', alpha=0.6, linewidth=2)

ax.set_xticks(np.arange(10) + bar_width*(len(periodos)-1)/2)
ax.set_xticklabels([f'D{d}' for d in range(1,11)], fontsize=12, fontweight='bold')
ax.set_ylabel('Tasa de Evento (%)', fontsize=14, fontweight='bold', color=COLOR_AZUL)
ax.set_title('Tasa de Evento por Decil (D10 = Mayor Score) | Backtest Out-of-Time', fontsize=16, fontweight='bold', color=COLOR_AZUL, pad=15)
ax.legend(fontsize=13, loc='upper left', framealpha=0.9)
ax.grid(axis='y', alpha=0.3, linestyle='--'); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show(); plt.close()

# ============================================================================
# GRÁFICO 3: CURVA DE CAPTURA ACUMULADA (desde D10 hacia D1)
# ============================================================================
fig, ax = plt.subplots(figsize=(14, 8))
fig.patch.set_facecolor('white'); ax.set_facecolor('white')

for i, per in enumerate(periodos):
    df_r = renumerar_deciles(resultados_por_periodo[per])
    # Captura acumulada desde D10 (mejor) hacia D1 (peor)
    total_ev = df_r['desembolsos'].sum()
    total_n = df_r['total'].sum()
    # Recalcular acumulado desde D10
    df_sorted = df_r.sort_values('decil_display', ascending=False)  # D10 primero
    pct_total_acum = (df_sorted['total'].cumsum() / total_n * 100).values
    pct_ev_acum = (df_sorted['desembolsos'].cumsum() / total_ev * 100).values
    
    ks_val = kpi_map[per]['ks_max']
    ax.plot(pct_total_acum, pct_ev_acum, marker='o', markersize=10, linewidth=3.5, color=COLORES_PERIODO[i],
            label=f'{label_map[per]} (KS={ks_val:.1f})', zorder=5)
    for x_v, y_v in zip(pct_total_acum, pct_ev_acum):
        ax.annotate(f'{y_v:.0f}%', xy=(x_v, y_v), xytext=(5,8), textcoords='offset points', fontsize=9, fontweight='bold', color=COLORES_PERIODO[i])

ax.plot([0,100],[0,100], 'k--', linewidth=2, alpha=0.4, label='Modelo aleatorio')
ax.set_xlabel('% Población (acum. desde D10)', fontsize=14, fontweight='bold', color=COLOR_AZUL)
ax.set_ylabel('% Eventos capturados (acum.)', fontsize=14, fontweight='bold', color=COLOR_AZUL)
ax.set_title('Curva de Captura Acumulada | Backtest Out-of-Time', fontsize=16, fontweight='bold', color=COLOR_AZUL, pad=15)
ax.legend(fontsize=13, loc='lower right', framealpha=0.9)
ax.set_xlim(0, 105); ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3, linestyle='--'); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show(); plt.close()

# ============================================================================
# GRÁFICO 4: LIFT POR DECIL (D10=mejor)
# ============================================================================
fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor('white'); ax.set_facecolor('white')

for i, per in enumerate(periodos):
    df_r = renumerar_deciles(resultados_por_periodo[per])
    lifts = df_r['lift'].values
    ax.plot(range(1,11), lifts, marker='s', markersize=12, linewidth=3.5, color=COLORES_PERIODO[i], label=label_map[per], zorder=5)
    for d, l in zip(range(1,11), lifts):
        ax.annotate(f'{l:.2f}x', xy=(d,l), xytext=(0,12), textcoords='offset points', ha='center',
                    fontsize=10, fontweight='bold', color=COLORES_PERIODO[i],
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=COLORES_PERIODO[i], alpha=0.8))

ax.axhline(y=1.0, color='red', linestyle='--', linewidth=2.5, alpha=0.6, label='Lift = 1 (aleatorio)')
ax.set_xticks(range(1,11)); ax.set_xticklabels([f'D{d}' for d in range(1,11)], fontsize=13, fontweight='bold')
ax.set_ylabel('Lift', fontsize=14, fontweight='bold', color=COLOR_AZUL)
ax.set_title('Lift por Decil (D10 = Mayor Score) | Backtest Out-of-Time', fontsize=16, fontweight='bold', color=COLOR_AZUL, pad=15)
ax.legend(fontsize=13, loc='upper left', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--'); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show(); plt.close()

# ============================================================================
# GRÁFICO 5: TABLA DETALLADA POR PERIODO (D10=mejor)
# ============================================================================
for per in periodos:
    df_r = renumerar_deciles(resultados_por_periodo[per])
    df_show = df_r[['decil_display','total','desembolsos','score_min','score_max','efectividad_desembolso','lift','ks']].copy()
    df_show.columns = ['Decil','Total','Eventos','Score Min','Score Max','Tasa_%','Lift','KS']
    
    fig, ax = plt.subplots(figsize=(18, 5))
    fig.patch.set_facecolor('white')
    ax.axis('off')
    
    cell_text = []
    for _, row in df_show.iterrows():
        cell_text.append([
            f'{int(row["Decil"])}',
            f'{int(row["Total"]):,}',
            f'{int(row["Eventos"]):,}',
            f'{row["Score Min"]:.3f}',
            f'{row["Score Max"]:.3f}',
            f'{row["Tasa_%"]:.2f}%',
            f'{row["Lift"]:.2f}x',
            f'{row["KS"]:.2f}'
        ])
    
    table = ax.table(cellText=cell_text, colLabels=['Decil','Total','Eventos','Score Min','Score Max','Tasa %','Lift','KS'],
                     cellLoc='center', loc='center', colColours=[COLOR_AZUL]*8)
    table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1.2, 1.8)
    
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(fontweight='bold', color='white')
            cell.set_facecolor(COLOR_AZUL)
        elif row == 10:  # D10 = mejor
            cell.set_facecolor((0.85, 1.0, 0.85))
            cell.set_text_props(fontweight='bold')
        elif row == 1:  # D1 = peor
            cell.set_facecolor((1.0, 0.9, 0.9))
        else:
            cell.set_facecolor('white')
        cell.set_edgecolor(COLOR_AZUL)
    
    tasa_g = kpi_map[per]['efectividad_global']
    ks_m = kpi_map[per]['ks_max']
    ax.set_title(f'Tabla de Deciles - {label_map[per]} | Tasa Global: {tasa_g:.2f}% | KS: {ks_m:.1f} | D10=Mayor Score',
                 fontsize=15, fontweight='bold', color=COLOR_AZUL, pad=20)
    plt.tight_layout(); plt.show(); plt.close()

# ============================================================================
# CONCLUSIÓN
# ============================================================================
print("\n" + "="*110)
print(" RESUMEN BACKTEST OUT-OF-TIME  (D10 = Mayor Score = Mayor Probabilidad)")
print("="*110)
for per in periodos:
    row = kpi_map[per]
    # efectividad_decil_1 en la data = mejor score → es nuestro D10
    # efectividad_decil_10 en la data = peor score → es nuestro D1
    tasa_d10 = row['efectividad_decil_1']
    tasa_d1 = row['efectividad_decil_10']
    lift_d10 = row['lift_decil_1']
    print(f"\n  Periodo: {label_map[per]} | N={int(row['total_registros']):,} | Eventos={int(row['total_desembolsos']):,}")
    print(f"    Tasa Global:  {row['efectividad_global']:.2f}%")
    print(f"    KS Máximo:    {row['ks_max']:.2f}")
    print(f"    D10 (mejor):  Tasa={tasa_d10:.2f}% | Lift={lift_d10:.2f}x")
    print(f"    D1  (peor):   Tasa={tasa_d1:.2f}%")
    ratio = tasa_d10 / tasa_d1 if tasa_d1 > 0 else 0
    print(f"    Ratio D10/D1: {ratio:.1f}x (separación entre mejor y peor decil)")

print(f"\n{'─'*110}")
print(f"  CONCLUSIÓN: Modelo estable con KS entre {resumen_periodos['ks_max'].min():.1f} y {resumen_periodos['ks_max'].max():.1f}")
lift_min = resumen_periodos['lift_decil_1'].min()
lift_max = resumen_periodos['lift_decil_1'].max()
print(f"  Lift D10 consistente: {lift_min:.2f}x - {lift_max:.2f}x")
print(f"  El D10 captura ~{resumen_periodos['efectividad_decil_1'].mean()/resumen_periodos['efectividad_global'].mean():.1f}x más eventos que el promedio")
print("="*110)

### 16. Escalera de propensión

In [ ]:
resumen_escalera, cortes_propension = escalera_propension(
    y_valid.values,
    resultados[mejor_modelo_nombre]['predicciones_valid'],
    n_grupos=10
)

### 17. Guardado de artefactos

In [ ]:
print("\n" + "="*100)
print(" PASO 12: GUARDAR MODELOS EN PKL")
print("="*100)

import joblib
import os
from datetime import datetime

# Carpeta de destino en S3 (ajustar ruta)
ruta_pkl_local = 'modelo_artifacts_3/'  # Ruta local temporal
os.makedirs(ruta_pkl_local, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# ------------------------------------------------------------
# Guardar TODOS los modelos entrenados
# ------------------------------------------------------------
print("\n Guardando modelos entrenados...")

for nombre, modelo in modelos_entrenados.items():
    # Limpiar nombre para usarlo como nombre de archivo
    nombre_archivo = nombre.replace(' ', '_').replace('(', '').replace(')', '').lower()
    archivo_pkl = f"{ruta_pkl_local}modelo_{nombre_archivo}_{timestamp}.pkl"
    
    joblib.dump(modelo, archivo_pkl)
    

    
    print(f"   ✓ {nombre:35s} → modelo_{nombre_archivo}_{timestamp}.pkl")

In [ ]:
print("\n Guardando mejor modelo con nombre fijo...")

nombre_mejor_archivo = mejor_modelo_nombre.replace(' ', '_').replace('(', '').replace(')', '').lower()
archivo_mejor_pkl = f"{ruta_pkl_local}mejor_modelo_{nombre_mejor_archivo}.pkl"

joblib.dump(mejor_modelo, archivo_mejor_pkl)


print(f"   ✓ Mejor modelo guardado como: mejor_modelo_{nombre_mejor_archivo}.pkl")

In [ ]:
print("\n Guardando metadata del modelo...")

metadata = {
    'mejor_modelo': mejor_modelo_nombre,
    'timestamp': timestamp,
    'variables_finales': variables_finales,
    'target': target,
    'metricas': {
        'AUC_train': float(df_comparacion.loc[mejor_modelo_nombre, 'AUC_train']),
        'AUC_valid': float(df_comparacion.loc[mejor_modelo_nombre, 'AUC_valid']),
        'AUC_test': float(df_comparacion.loc[mejor_modelo_nombre, 'AUC_test']),
        'Gini_test': float(df_comparacion.loc[mejor_modelo_nombre, 'Gini_test']),
        'KS_test': float(resultados[mejor_modelo_nombre]['KS_test'])
    },
    'n_trials_optuna': N_TRIALS_OPTUNA,
    'seed': SEED
}

archivo_metadata = f"{ruta_pkl_local}metadata_modelo_{timestamp}.pkl"
joblib.dump(metadata, archivo_metadata)


print(f"   ✓ Metadata guardada con métricas y parámetros")